##PYSPARK

In [0]:
df = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("/Volumes/data_engineer/bronze/bronze_volume/Customers/dim_customers.csv")
display(df)

In [0]:
from pyspark.sql.functions import upper, col
df = df.withColumn("name", upper(col("name")))
display(df)

In [0]:
from pyspark.sql.functions import *
df = df.withColumn("domain", split(col("email"), "@")[1])
display(df)

In [0]:
display(df.groupBy("domain").agg(count("domain").alias("Count_domain"))\
        .sort(col("Count_domain").desc()))

Databricks visualization. Run in Databricks to view.

###**Upsert the data Update and insert**###

In [0]:
from delta.tables import DeltaTable
if spark.catalog.tableExists("data_engineer.silver.customer_enr"):
    dlt_obj = DeltaTable.forName(spark,"data_engineer.silver.customer_enr")
    dlt_obj.alias("trgt").merge(df.alias("src"), "trgt.customer_id = src.customer_id")\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll() 
    
else:
    df.write.format("delta")\
        .mode("append")\
        .saveAsTable("data_engineer.silver.customer_enr")

###**Products**###

In [0]:
df_prod = spark.read.format("csv")\
            .option("header", "true")\
            .option("inferSchema", "true")\
            .load("/Volumes/data_engineer/bronze/bronze_volume/products/dim_products.csv")
display(df_prod)

In [0]:
from pyspark.sql import *
from pyspark.sql.functions import *
df_prod = df_prod.withColumn("Processed_Date",current_timestamp())
display(df_prod)

In [0]:
display(df_prod.groupBy("category").agg(avg("price").alias("avg_price"))\
    .sort(col("avg_price").desc()))

Databricks visualization. Run in Databricks to view.

In [0]:
from delta.tables import DeltaTable
if spark.catalog.tableExists("data_engineer.silver.product_enr"):
    dlt_obj = DeltaTable.forName(spark,"data_engineer.silver.product_enr")
    dlt_obj.alias("trgt").merge(df_prod.alias("src"), "trgt.product_id = src.product_id")\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll() 
    
else:
    df_prod.write.format("delta")\
        .mode("append")\
        .saveAsTable("data_engineer.silver.product_enr")

In [0]:
%sql
select * from data_engineer.silver.product_enr order by product_id

###**STORES**###

In [0]:
df_st = spark.read.format("csv")\
            .option("header", "true")\
            .option("inferSchema", "true")\
            .load("/Volumes/data_engineer/bronze/bronze_volume/stores/dim_stores.csv")
display(df_st)

In [0]:
from pyspark.sql.functions import *
df_st = df_st.withColumn("store_name",regexp_replace(col("store_name"),"_"," "))
display(df_st)


In [0]:
df_st = df_st.withColumn("Processed_Date",current_timestamp())
display(df_st)

In [0]:
from delta.tables import DeltaTable
if spark.catalog.tableExists("data_engineer.silver.store_enr"):
    dlt_obj = DeltaTable.forName(spark,"data_engineer.silver.store_enr")
    dlt_obj.alias("trgt").merge(df_st.alias("src"), "trgt.store_id = src.store_id")\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()
else:
    df_st.write.format("delta")\
        .mode("append")\
        .saveAsTable("data_engineer.silver.store_enr")

In [0]:
%sql
select * from data_engineer.silver.store_enr

##**SALES**

In [0]:
df_sl = spark.read.format("csv")\
        .option("header", "true")\
        .option("inferSchema", "true")\
        .load("/Volumes/data_engineer/bronze/bronze_volume/sales/fact_sales_3.csv")
display(df_sl)


In [0]:
from pyspark.sql.functions import *
from pyspark.sql import *
df_sl = df_sl.withColumn("pricePerSale",round(col("total_amount")/col("quantity"),3))
df_sl = df_sl.withColumn("Processed_Date",current_timestamp())
display(df_sl)

In [0]:
from delta.tables import DeltaTable
if spark.catalog.tableExists("data_engineer.silver.sales_enr"):
    dlt_obj = DeltaTable.forName(spark,"data_engineer.silver.sales_enr")
    dlt_obj.alias("trgt").merge(df_sl.alias("src"), "trgt.sales_id = src.sales_id")\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()
else:
    df_sl.write.format("delta")\
        .mode("append")\
        .saveAsTable("data_engineer.silver.sales_enr")

In [0]:
%sql
select * from data_engineer.silver.sales_enr

###**SPARK SQL**

In [0]:
df_sl.createOrReplaceTempView("tempSales")

In [0]:
display(spark.sql("""SELECT * FROM tempSales"""))

###PYSPARK UDF

In [0]:
def greet(name):
  return f"Hello {name}!"

In [0]:
udf_greet = udf(greet)
display(df)

In [0]:
display(df.withColumn("greet",udf_greet(col("name"))))